# Unrolled ADMM convergence video (8 iterations)

Runs the trained 8-iteration PnP Unrolled-ADMM (`Gaussian_PnP_ADMM_8iters_MultiScale_38epochs.pth`) on the tutorial galaxy observation and records the per-layer deconvolution output. The video starts at the raw observation (iteration 0) and advances one frame per unrolled layer, giving `n_iters + 1` frames. Left: observation. Right: current ADMM iterate.

In [ ]:
import os
import sys

sys.path.insert(0, '../../')

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video
from matplotlib import animation

from models.unrolled_admm_gaussian import UnrolledADMMGaussian

FONT_WEIGHT = 'bold'
plt.rcParams.update({'font.weight': FONT_WEIGHT})
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
n_iters = 8
best_epoch = 38
snr = 100
FPS = 3
HOLD_LAST = 1.0        # seconds the final frame is held
SHOW_ITER = False      # print the iteration number on the right panel
CMAP = 'magma'
DATA_PATH = '../../tutorials/'
SAVED_MODEL_PATH = '../../saved_models_2024/'

### Data

In [ ]:
psf = torch.load(os.path.join(DATA_PATH, 'psf.pth'))
gt = torch.load(os.path.join(DATA_PATH, 'gt.pth'))
obs = torch.load(os.path.join(DATA_PATH, 'obs.pth'))
obs_np = obs.cpu().numpy() if isinstance(obs, torch.Tensor) else np.asarray(obs)

### Run Unrolled ADMM and record the trajectory

In [ ]:
from torch.fft import fft2, ifftshift
from utils.utils_torch import pad_double

model = UnrolledADMMGaussian(n_iters=n_iters, PnP=True, subnet=True, analysis=True).cuda()
model.eval()
model_file = os.path.join(SAVED_MODEL_PATH, f'Gaussian_PnP_ADMM_{n_iters}iters_MultiScale_{best_epoch}epochs.pth')
model.load_state_dict(torch.load(model_file, map_location=torch.device('cuda:0')))
print(f'Loaded {model_file}')

y = obs.unsqueeze(0).unsqueeze(0).cuda()
k = psf.unsqueeze(0).unsqueeze(0).cuda()
alpha = obs.ravel().mean().view(1, 1, 1, 1).cuda()

# Reproduce the model's internal Wiener/Tikhonov initialization (init_l2 in UnrolledADMMGaussian.forward).
with torch.no_grad():
    y_pos = torch.maximum(y, torch.zeros_like(y))
    Y = fft2(ifftshift(pad_double(y_pos), dim=(-2, -1)))
    H = fft2(ifftshift(pad_double(k), dim=(-2, -1)))
    Ht, HtH = torch.conj(H), torch.abs(H) ** 2
    z0 = model.init_l2(Y, Ht, HtH, alpha)
    x_list, z_list, u_list, rho_list = model(y, k, alpha)
z0_np = z0.cpu().squeeze().detach().numpy()
z_list = [z.cpu().squeeze().detach().numpy() for z in z_list]

# Frame 0: Wiener/Tikhonov initialization (the z fed into the first ADMM layer).
# Frames 1..n_iters: per-layer denoiser (Z-update) outputs — same space as the init.
# The last frame equals model(...) when analysis=False (returns z_list[-1]).
frames_data = [z0_np] + z_list
n_frames = len(frames_data)
print(f'{n_frames} frames (init, denoiser outputs 1 ... {n_iters})')

### Video

In [ ]:
# Match analysis.ipynb: no shared norm, each frame auto-scales to its own min/max.
# The left panel is always `obs`, so its colour range stays fixed; the right panel re-scales
# per iteration, matching the per-image normalization used by the original frame export.
fig, (ax_obs, ax_rec) = plt.subplots(1, 2, figsize=(10, 5))
fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0, wspace=0.04)

h_obs = ax_obs.imshow(obs_np, cmap=CMAP)
h_rec = ax_rec.imshow(frames_data[0], cmap=CMAP)
h_txt = ax_rec.text(4, 10, '', color='white', fontsize=16, fontweight=FONT_WEIGHT) if SHOW_ITER else None
for ax in (ax_obs, ax_rec):
    ax.set_axis_off()


def update(i):
    data = frames_data[i]
    h_rec.set_data(data)
    h_rec.set_clim(vmin=data.min(), vmax=data.max())
    if SHOW_ITER:
        h_txt.set_text('Observation' if i == 0 else f'Iteration {i}')
    return h_obs, h_rec


# Hold the final reconstruction for HOLD_LAST seconds by repeating its frame.
frames = list(range(n_frames)) + [n_frames - 1] * int(round(HOLD_LAST * FPS))
anim = animation.FuncAnimation(fig, update, frames=frames, blit=False)
video_path = f'unrolled_admm_8iters_SNR={snr}.mp4'
anim.save(video_path, writer=animation.FFMpegWriter(fps=FPS, bitrate=2400), dpi=150,
          savefig_kwargs={'facecolor': 'white'})
# The last frame (the final reconstruction) as a still image.
update(n_frames - 1)
image_path = f'unrolled_admm_8iters_SNR={snr}_last.png'
fig.savefig(image_path, dpi=150, facecolor='white')
plt.close(fig)
print('saved', video_path, 'and', image_path)
Video(video_path, embed=True)